# Pump failure example with Pareto prior

This notebook demonstrates the use of `MGFDerivative` with a Pareto prior for the Poisson likelihood, using the classic pump failure data from Gaver and O'Muircheartaigh (1987).

We will:
- Compute the model evidence (marginal likelihood)
- Compute the posterior density
- Compute the posterior predictive mass for a new observation
- Compute the posterior MGF and moments
- Demonstrate sequential updating by splitting the data into two chunks

All examples use the **single‑rate model** with a diffuse Pareto prior $ \Lambda \sim \text{Pareto}(\alpha=10^{-5}, \xi=10^{-5}) $. This matches the setup in the paper.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from jumufraktiv.MGFDerivative_class import MGFDerivative
from jumufraktiv.like_stats.Poisson import readyPoisson
from scipy.special import expn
import mpmath as mp
import numpy as np
import math

plt.style.use('seaborn-v0_8-whitegrid')

## Pump failure data

In [ ]:
data = pd.DataFrame({
    't': [94.32, 15.72, 62.88, 125.76, 5.24, 31.44, 1.048, 1.048, 2.096, 10.48],
    'y': [5, 1, 5, 14, 3, 19, 1, 1, 4, 22]
})
print("Data:")
display(data)

## Prior specification

We use a Pareto prior with shape `α = 1e-5` and scale `ξ = 1e-5` (very diffuse). Note that the Poisson likelihood gives an integer derivative order (`a = sum(y_i)`), so we will use `method='symbolic'` for exact computation.

## 1. Model evidence

The evidence is computed via the derivative of the prior MGF at `t = -b`. We compute it symbolically and compare with the analytic formula from the paper.

In [ ]:
# Compute evidence using the package
from jumufraktiv.MGFPrior_class import MGFPrior
import jumufraktiv.MGFdictionary  # registers priors

alpha_prior = 1e-5
xi_prior = 1e-5

# ---- Create Pareto prior object ----
pareto_prior = MGFPrior.from_registry(
    "pareto",
    params={"alpha": alpha_prior, "xi": xi_prior}
)

deriv = MGFDerivative(
    prior=pareto_prior,   # prior object
    data=data['y'],
    likelihood='poisson',
    method='symbolic',
    scale=data['t']       # likelihood parameter
)

log_ev_pkg = deriv.evidence()
print(f"Package log evidence: {log_ev_pkg:.6f}")

**1.2 Analytical verification using the paper's formula**

The evidence for the single‑rate Pareto‑Poisson model is given by Equation (14) in the paper:

$$
p(\mathbf{y}) = \left(\prod_{i=1}^n \frac{t_i^{y_i}}{y_i!}\right) \, \alpha \, \xi^{\sum y_i} \, E_{\alpha+1-\sum y_i}\left(\xi \sum t_i\right),
$$

where $E_n(z)$ is the exponential integral function. We can verify the package result by directly substituting the hyperparameters into the symbolic derivative expression obtained from the package.

In [ ]:
# Set high precision for accurate evaluation
mp.mp.dps = 50  # note: use mp.mp.dps, not mp.dps

# Compute statistics
a = sum(data['y'])
b = sum(data['t'])
c_val = np.prod(data['t']**data['y'] / [math.factorial(int(yi)) for yi in data['y']])

# Hyperparameters
alpha_prior = 1e-5
xi_prior = 1e-5

# Order and argument of the exponential integral
n_expint = alpha_prior + 1 - a
z_expint = xi_prior * b

# Evaluate E_n(z) using mpmath (supports negative n)
E_val = mp.expint(n_expint, z_expint)

# Compute log evidence
log_ev_analytic = np.log(c_val) + np.log(alpha_prior) + a * np.log(xi_prior) + np.log(float(E_val))
print(f"Analytical log evidence (mpmath): {log_ev_analytic:.6f}")

# Compare with package result (from previous chunk)
print(f"Package log evidence: {log_ev_pkg:.6f}")
print(f"Difference: {log_ev_analytic - log_ev_pkg:.2e}")

## 2. Posterior density

The posterior density is given by (16). We compute it over a range of λ values and plot.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---- Compute MLE ----
a = sum(data['y'])
b = sum(data['t'])
mle = a / b

# ---- Posterior density ----
lambda_vals = np.linspace(1e-6, 0.4, 500)
log_post = deriv.post_density(lambda_vals, log=True)
post = np.exp(log_post)

# ---- Plot ----
plt.figure(figsize=(8,5))
plt.plot(lambda_vals, post, label='Posterior density')
plt.axvline(x=mle, color='r', linestyle='--', label=f'MLE = {mle:.3f}')
plt.xlabel(r'$\lambda$')
plt.ylabel('Posterior density')
plt.title('Posterior density of the common failure rate')
plt.legend()
plt.grid()
plt.show()

In [ ]:
import sympy as sp
from jumufraktiv.MGFPrior_class import MGFPrior
from jumufraktiv.symbols import t, theta, param
from jumufraktiv.MGFDerivative_class import MGFDerivative

# ---- Define symbolic hyperparameters for Pareto ----
alpha_sym = param("alpha")
xi_sym = param("xi")

# ---- Create a symbolic Pareto prior ----
# `from_registry` compiles a NUMERIC density, so it cannot take symbols; a
# prior whose hyperparameters stay free is built directly. The Pareto MGF is
# alpha * E_{alpha+1}(-xi t), with density alpha * xi**alpha / theta**(alpha+1).
from sympy.functions.special.error_functions import expint

symbolic_prior = MGFPrior(
    name="pareto_symbolic",
    mgf_sym=alpha_sym * expint(alpha_sym + 1, -xi_sym * t),
    pdf_sym=alpha_sym * xi_sym**alpha_sym / theta**(alpha_sym + 1),
    params={},
).as_MGFPrior()

# ---- Pass to MGFDerivative ----
# Use the same data (e.g., pump failure data)
deriv_sym = MGFDerivative(
    prior=symbolic_prior,
    data=data['y'],
    likelihood='poisson',
    method='auto',
    scale=data['t']
)

# ---- Check result ----
print(f"deriv_sym.is_symbolic = {deriv_sym.is_symbolic}")   # Should be True

# ---- Compute symbolic posterior density ----
post_expr = deriv_sym.post_density(theta, log=False)
print("Symbolic posterior density (Pareto prior):")
sp.pprint(post_expr, use_unicode=False)

## 3. Posterior cumulative density function

In [ ]:
# ---- Posterior CDF (vectorised) ----
u_vals = np.linspace(1e-6, 0.4, 500)   # use full grid for smooth plot

# post_cdf now accepts an array of u values and returns an array of log‑CDFs
log_cdf = deriv.post_cdf(u_vals, log=True)
cdf = np.exp(log_cdf)

print("Posterior CDF values (first 10):")
for u, c in zip(u_vals[:10], cdf[:10]):
    print(f"u = {u:.4f}, CDF = {c:.4f}")

# ---- Plot ----
plt.figure(figsize=(8,5))
plt.plot(u_vals, cdf, label='Posterior CDF', linewidth=2)
plt.axvline(x=mle, color='r', linestyle='--', label=f'MLE = {mle:.3f}')
plt.xlabel(r'$\lambda$')
plt.ylabel(r'Posterior CDF $F_{\Lambda|y}(u)$')
plt.title('Posterior CDF of the common failure rate')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

# ---- Median (vectorised search) ----
idx = np.searchsorted(cdf, 0.5)
if idx < len(u_vals):
    # Linear interpolation for more accurate median
    if idx == 0:
        median = u_vals[0]
    elif idx == len(u_vals):
        median = u_vals[-1]
    else:
        # Interpolate between u_vals[idx-1] and u_vals[idx]
        u0, u1 = u_vals[idx-1], u_vals[idx]
        c0, c1 = cdf[idx-1], cdf[idx]
        median = u0 + (0.5 - c0) * (u1 - u0) / (c1 - c0)
    print(f"Posterior median ≈ {median:.4f}")

## 4. Posterior credible interval

The plot below visualises the posterior distribution of the common failure rate $\lambda$ for the single‑rate Pareto‑Poisson model. It combines:

1. **Posterior density** (solid black curve) – computed via the vectorised `post_density` method.
2. **Central credible intervals** (shaded regions and vertical dashed lines) – equal‑tailed intervals at 68%, 95%, and 99% credible levels, obtained using the vectorised `post_interval` method (which relies on `post_quantile`).
3. **Maximum likelihood estimate (MLE)** – $\hat{\lambda} = a / b$, shown as a red vertical line.

The shaded areas under the density correspond to the central credible intervals; for example, the 95% interval covers the central 95% of the posterior mass.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---- Compute MLE ----
a = sum(data['y'])
b = sum(data['t'])
mle = a / b

# ---- Posterior density ----
lambda_vals = np.linspace(1e-6, 0.4, 500)
log_post = deriv.post_density(lambda_vals, log=True)
post = np.exp(log_post)

# ---- Compute credible intervals ----
levels = [0.68, 0.95, 0.99]
intervals = deriv.post_interval(level=levels, verbose=True)  # shape (3, 2)
colors = ['green', 'blue', 'red']
labels = [f'{int(l*100)}% CI' for l in levels]

# ---- Print intervals table ----
print("Credible Intervals:")
print("Level       Lower        Upper")
print("--------------------------------")
for level, (lower, upper) in zip(levels, intervals):
    print(f"{level*100:3.0f}%      {lower:.4f}      {upper:.4f}")

# ---- Plot with shaded credible intervals ----
plt.figure(figsize=(10, 6))

# Density
plt.plot(lambda_vals, post, 'k-', linewidth=2, label='Posterior density')

# Credible intervals: shaded under the density curve
for (lower, upper), color, label in zip(intervals, colors, labels):
    mask = (lambda_vals >= lower) & (lambda_vals <= upper)
    plt.fill_between(lambda_vals[mask], post[mask], alpha=0.25, color=color, label=label)
    plt.axvline(lower, color=color, linestyle='--', alpha=0.7)
    plt.axvline(upper, color=color, linestyle='--', alpha=0.7)

# MLE
plt.axvline(x=mle, color='r', linestyle='-', linewidth=2, label=f'MLE = {mle:.3f}')

# Labels and legend
plt.xlabel(r'$\lambda$')
plt.ylabel('Posterior density')
plt.title('Posterior density with credible intervals')
plt.legend(loc='best')
plt.grid(alpha=0.3)
plt.xlim(0, 0.3)
plt.show()

## 5. Posterior sample

**Posterior sample** – a rug plot of 200 draws from the posterior, generated via inverse transform sampling (`post_sample`). This chunk generates 200 posterior samples via inverse transform sampling (`post_sample`) and overlays a normalised histogram of the samples on the posterior density curve. The histogram provides a non‑parametric check of the density shape and allows visual assessment of the sample quality.

In [ ]:
# ---- Generate posterior samples ----
samples = deriv.post_sample(n=200, verbose=True)

# ---- Plot ----
plt.figure(figsize=(10, 6))

# Density
plt.plot(lambda_vals, post, 'k-', linewidth=2, label='Posterior density')

# MLE
plt.axvline(x=mle, color='r', linestyle='-', linewidth=2, label=f'MLE = {mle:.3f}')

# Histogram of samples (normalised)
plt.hist(samples, bins=30, density=True, alpha=0.5, color='gray', label='Posterior samples (hist)')

# Labels and legend
plt.xlabel(r'$\lambda$')
plt.ylabel('Density / frequency')
plt.title('Posterior density with histogram of samples')
plt.legend(loc='best')
plt.grid(alpha=0.3)
plt.xlim(0, 0.3)
plt.show()

## 6. Posterior predictive mass

We compute the predictive mass for a new observation at pump 4's exposure (`t_new = 125.76`) and compare with the observed `y=14`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- Parameters for pump 4 ----
t_new = 125.76
y_obs = 14

# ---- Compute predictive mass for y = 0..50 (vectorised) ----
y_vals = np.arange(0, 51)

# Vectorized call: pass the array of y values and the scale parameter.
# The method returns an array of log-predictive masses.
log_preds = deriv.post_predictive(
    y_vals,          # array of y values
    scale=t_new,     # exposure for Poisson
    log=True,
    individual=True
)
pred_masses = np.exp(log_preds)   # now an array of masses

# ---- Plot ----
plt.figure(figsize=(10, 6))
plt.bar(y_vals, pred_masses, width=0.8, alpha=0.7, label='Predictive mass')
plt.axvline(x=y_obs, color='r', linestyle='--', linewidth=2, label=f'Observed y = {y_obs}')
plt.xlabel('Number of failures (y)')
plt.ylabel('Posterior predictive probability mass')
plt.title(f'Posterior predictive mass for pump with t = {t_new}')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# ---- Also print the predictive mass at the observed value ----
# Scalar input returns a scalar (not an array)
log_pred_obs = deriv.post_predictive(
    y_obs,
    scale=t_new,
    log=True,
    individual=True
)
print(f"Predictive mass at y={y_obs}: {np.exp(log_pred_obs):.6e}")

In [ ]:
import pandas as pd
import sympy as sp
import numpy as np
from jumufraktiv.symbols import t, theta, param
from jumufraktiv.MGFDerivative_class import MGFDerivative

# ---- Assume deriv_sym is a symbolic MGFDerivative object ----
# It should have is_symbolic = True and contain symbolic hyperparameters.

# ---- New observations ----
y_vals = [14, 15, 16, 17, 18, 19]   # test a few y values
t_new = 125.76

# ---- Compute symbolic predictive masses (individual=True) ----
pred_mass_results = deriv_sym.post_predictive(
    y_vals,               # array-like of y values
    scale=t_new,
    log=False,            # return ordinary density (symbolic expressions)
    individual=True       # return per-element results
)

print("Symbolic posterior predictive mass expressions for y = 14..19:")
for y, expr in zip(y_vals, pred_mass_results):
    print(f"y = {y}:")
    sp.pprint(expr, use_unicode=False)
    print()


## 7. Posterior MGF

The posterior MGF as a function of `r` (for `r < 0`):

In [ ]:
r_vals = np.linspace(-1e3-100, 350, 300)
cgf_vals = deriv.post_mgf(r_vals, log=True)   # vectorized: returns array

plt.figure(figsize=(8,5))
plt.plot(r_vals, cgf_vals)
plt.xlabel('r')
plt.ylabel('Posterior CGF')
plt.title('Posterior CGF of the rate parameter')
plt.grid()
plt.show()

In [ ]:
import sympy as sp
from jumufraktiv.symbols import param

# ---- Define symbolic variable for r (the MGF argument) ----
r_sym = sp.Symbol('r', real=True)

# ---- Compute symbolic posterior MGF (ordinary scale) ----
post_mgf_expr = deriv_sym.post_mgf(r_sym, log=False)

print("Symbolic posterior MGF (ordinary scale):")
sp.pprint(post_mgf_expr, use_unicode=False)

# ---- Compute symbolic posterior log‑MGF (CGF) ----
post_cgf_expr = deriv_sym.post_mgf(r_sym, log=True)

print("\nSymbolic posterior CGF (log MGF):")
sp.pprint(post_cgf_expr, use_unicode=False)

## 8. Posterior raw moments

We compute the first few moments (mean, variance, etc.) using the `post_moment` method. As 'symbolic' method does not work well on fractional orders of derivatives, we focus on integer-ordered moments only.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---- Compute sufficient statistics ----
a = sum(data['y'])        # Σ y_i (e.g., 75)

# ---- Define order range ----
q_min = -int(a)           # e.g., -75
q_max = 1100

# ---- Select 100 evenly spaced integers between q_min and q_max ----
q_vals = np.unique(np.round(np.linspace(q_min, q_max, 10)).astype(int))
# Ensure q_min and q_max are included (just in case)
if q_min not in q_vals:
    q_vals = np.append(q_min, q_vals)
if q_max not in q_vals:
    q_vals = np.append(q_vals, q_max)
q_vals = np.sort(q_vals)

print(f"Number of selected orders: {len(q_vals)}")
print(f"Selected orders (q): {q_vals}")

# ---- Compute log|E[Θ^q]| for all selected orders (vectorised) ----
log_abs_moments_raw = deriv.post_raw_moment(q_vals, log=True, numerator_method='symbolic')

# Convert to float array (handle symbolic results as NaN)
log_abs_moments = np.full(len(q_vals), np.nan, dtype=float)
for i, val in enumerate(log_abs_moments_raw):
    try:
        log_abs_moments[i] = float(val)
    except (TypeError, ValueError):
        # Keep NaN if not convertible to float
        pass

# ---- Plot ----
plt.figure(figsize=(12, 6))
plt.plot(q_vals, log_abs_moments, linestyle='-', linewidth=1.5, label='log|E[Θ^q]| (sampled)')
plt.axvline(x=0, color='black', linestyle='--', alpha=0.5, label='q = 0')
plt.xlabel('Order q (integer)')
plt.ylabel('log|E[Θ^q]|')
plt.title('Posterior moments of the rate parameter (sampled integer orders)')
plt.legend()
plt.grid(alpha=0.3)
plt.xlim(q_min, q_max)
plt.show()

As SymPy does not support differentiations up to symbolic orders, the symbolic method to calculate posterior moments is not supported.

In [ ]:
import sympy as sp

# ---- Define symbolic variable for the moment order ----
q_sym = sp.Symbol('q', real=True)

# `sp.diff(expr, t, n)` needs a concrete n, so a symbolic order is refused
# rather than silently approximated. The refusal is shown here, not raised,
# so the rest of the notebook still runs.
for scale, label in [(False, "E[theta^q] (ordinary scale)"),
                     (True, "log|E[theta^q]| (log scale)")]:
    try:
        expr = deriv_sym.post_raw_moment(q_sym, log=scale)
        print(f"Symbolic posterior moment {label}:")
        sp.pprint(expr, use_unicode=False)
    except NotImplementedError as error:
        print(f"{label}: refused -- {error}")

## 9. Posterior central moments

In [ ]:
print("=" * 60)
print("Testing central moments (orders 1–4)")
print("=" * 60)

# ---- For each order, compute both log and ordinary ----
for order in [1, 2, 3, 4]:
    try:
        # Log‑scale: returns (log_abs, sign)
        log_abs, sign = deriv.post_central_moment(order, log=True, numerator_method='symbolic')
        print(f"Central moment {order} (log-scale):")
        if isinstance(log_abs, sp.Expr):
            print(f"  log|.| = symbolic expression:")
            sp.pprint(log_abs, use_unicode=False)
            print(f"  sign = {sign}")
        else:
            print(f"  log|.| = {log_abs:.6f}, sign = {sign}")
    except Exception as e:
        print(f"  Log‑scale computation failed for order {order}: {e}")

    try:
        # Ordinary scale
        val = deriv.post_central_moment(order, log=False, numerator_method='symbolic')
        if isinstance(val, sp.Expr):
            print(f"Central moment {order} (ordinary, symbolic):")
            sp.pprint(val, use_unicode=False)
        else:
            print(f"Central moment {order} (ordinary, numeric): {val:.6e}")
    except Exception as e:
        print(f"  Ordinary‑scale computation failed for order {order}: {e}")

    print("-" * 40)

## 10. Sequential updating

We split the data into two chunks: first 5 pumps, then the remaining 5. We update sequentially and compare the final evidence with the full-dataset evidence.

### symbolic-symbolic

In [ ]:
import time
from jumufraktiv.MGFPrior_class import MGFPrior
import jumufraktiv.MGFdictionary
from jumufraktiv.MGFDerivative_class import MGFDerivative

# ---- Create non‑informative Gamma prior ----
alpha_prior = 1e-5
beta_prior = 1e-5
gamma_prior = MGFPrior.from_registry(
    "gamma",
    params={"alpha": alpha_prior, "beta": beta_prior}
)

# ---- Split data ----
data1 = data.iloc[:5]
data2 = data.iloc[5:]

# ---- First chunk ----
deriv1 = MGFDerivative(
    prior=pareto_prior,
    data=data1['y'],
    likelihood='poisson',
    method='auto',
    scale=data1['t']
)

# ---- Investigate deriv1 ----
print(f"deriv1._is_symbolic = {deriv1._is_symbolic}")
print(f"deriv1._deriv_is_symbolic = {deriv1._deriv_is_symbolic}")

# ---- Now call update() ----
deriv2 = deriv1.update(
    new_data=data2['y'],
    method='auto',
    scale=data2['t']
)

# ---- Investigate deriv2 ----
print(f"\nderiv2._is_symbolic = {deriv2._is_symbolic}")
print(f"deriv2._deriv_is_symbolic = {deriv2._deriv_is_symbolic}")
print(f"deriv2.prior.mgf_sym is None? {deriv2.prior.mgf_sym is None}")

# ---- Also check the prior's backend ----
print(f"deriv2.prior.mgf_backend is None? {deriv2.prior.mgf_backend is None}")

### model evidence

In [ ]:
log_ev_pkg1 = deriv1.evidence()
print(f"Package log evidence by sequential update: {log_ev_pkg1:.6f}")

log_ev_pkg2 = deriv2.evidence()
print(f"Package log evidence by sequential update: {log_ev_pkg2:.6f}")

print(f"Total evidence: {log_ev_pkg1 + log_ev_pkg2:.2e}")
print(f"Difference with first evidence: {log_ev_pkg1 + log_ev_pkg2 - log_ev_pkg:.2e}")

### posterior density

In [ ]:
log_post2 = deriv2.post_density(lambda_vals, log=True)
post2 = np.exp(log_post2)

# ---- Plot ----
plt.figure(figsize=(8,5))
plt.plot(lambda_vals, post2, label='Posterior density')
plt.axvline(x=mle, color='r', linestyle='--', label=f'MLE = {mle:.3f}')
plt.xlabel(r'$\lambda$')
plt.ylabel('Posterior density')
plt.title('Posterior density of failure rate')
plt.suptitle('by sequential updating')
plt.legend()
plt.grid()
plt.show()

### posterior predictive

In [ ]:
# ---- Compute predictive mass for y = 0..50 (vectorised) ----
y_vals = np.arange(0, 51)

# Vectorized call: pass the array of y values and the scale parameter.
# The method returns an array of log-predictive masses.
log_preds_vec = deriv2.post_predictive(
    y_vals,          # array of y values
    scale=t_new,     # exposure for Poisson
    log=True,
    individual=True
)
pred_masses2 = np.exp(log_preds_vec)   # now an array of masses

# ---- Plot ----
plt.figure(figsize=(10, 6))
plt.bar(y_vals, pred_masses2, width=0.8, alpha=0.7, label='Predictive mass')
plt.axvline(x=y_obs, color='r', linestyle='--', linewidth=2, label=f'Observed y = {y_obs}')
plt.xlabel('Number of failures (y)')
plt.ylabel('Posterior predictive probability mass')
plt.title(f'Posterior predictive mass for pump with t = {t_new}')
plt.suptitle('by sequential updating')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# ---- Also print the predictive mass at the observed value ----
# Scalar input returns a scalar (not an array)
log_pred_obs = deriv2.post_predictive(
    y_obs,
    scale=t_new,
    log=True,
    individual=True
)
print(f"Predictive mass at y={y_obs}: {np.exp(log_pred_obs):.6e}")

### posterior MGF

In [ ]:
cgf_vals2 = deriv2.post_mgf(r_vals, log=True)   # vectorized: returns array

plt.figure(figsize=(8,5))
plt.plot(r_vals, cgf_vals2)
plt.xlabel('r')
plt.ylabel('Posterior CGF')
plt.title('Posterior CGF of the rate parameter')
plt.suptitle('by sequential updating')
plt.grid()
plt.show()

### posterior moments
Sequential updating sometimes narrows down the range of which the posterior moments are defined. In this example, the negative posterior moments are defined up to -a, the differentiation order. However, the 'a' in the sequential update is twofold: the order determined by the first or second set of data; only the latter determines the boundary of the order of negative posterior moments.

In [ ]:
print("lower bound of negative posterior moments in one-step inference", -deriv.a)
print("lower bound of negative posterior moments in sequential updates", -deriv2.a)

Asking for order $q = -75$ is therefore fine against `deriv` and out of range against `deriv2`. The cell below catches the refusal and prints it, so the contrast is visible without stopping the notebook.

In [ ]:
# Order q = -75 is admissible against `deriv` (a = 75, so the derivative order
# a + q is exactly 0) and inadmissible against `deriv2` (a = 47, so a + q < 0).
# The refusal is caught rather than raised, so the notebook runs to the end.
for label, obj in [("one-step", deriv), ("sequential", deriv2)]:
    try:
        value = obj.post_raw_moment(int(-75), log=True, numerator_method='symbolic')
        print(f"{label} (a = {obj.a:g}), q = -75: log|E[Theta^q]| = {value}")
    except ValueError as error:
        print(f"{label} (a = {obj.a:g}), q = -75: refused -- {error}")

In [ ]:
# ---- Define order range ----
q_min2 = -int(deriv2.a)       # e.g., -75
q_max2 = 1100

# ---- Select 10 evenly spaced integers between q_min and q_max ----
q_vals2 = np.unique(np.round(np.linspace(q_min2, q_max2, 10)).astype(int))
# Ensure q_min and q_max are included (just in case)
if q_min2 not in q_vals2:
    q_vals2 = np.append(q_min2, q_vals2)
if q_max2 not in q_vals2:
    q_vals2 = np.append(q_vals2, q_max2)
q_vals2 = np.sort(q_vals2)

print(f"Number of selected orders: {len(q_vals2)}")
print(f"Selected orders (q): {q_vals2}")

# ---- Compute log|E[Θ^q]| for all selected orders (vectorised) ----
log_abs_moments_raw2 = deriv2.post_raw_moment(q_vals2, log=True, numerator_method='symbolic')

# Convert to float array (handle symbolic results as NaN)
log_abs_moments2 = np.full(len(q_vals2), np.nan, dtype=float)
for i, val in enumerate(log_abs_moments_raw2):
    try:
        log_abs_moments2[i] = float(val)
    except (TypeError, ValueError):
        # Keep NaN if not convertible to float
        pass

# ---- Plot ----
plt.figure(figsize=(12, 6))
plt.plot(q_vals2, log_abs_moments2, linestyle='-', linewidth=1.5, label='log|E[Θ^q]| (sampled)')
plt.axvline(x=0, color='black', linestyle='--', alpha=0.5, label='q = 0')
plt.xlabel('Order q (integer)')
plt.ylabel('log|E[Θ^q]|')
plt.title('Posterior moments of the rate parameter (sampled integer orders)')
plt.suptitle('by sequential updating')
plt.legend()
plt.grid(alpha=0.3)
plt.xlim(q_min2, q_max2)
plt.show()